# Compare two evaluators

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Xmaster6y/lczerolens/blob/main/docs/source/notebooks/tutorials/compare-models.ipynb)

This tutorial compares two deterministic fixture networks across the same position set. The result is an observational policy/value comparison with frozen provenance—not a strength estimate or a causal explanation. Replace the fixtures with pinned model paths for a real study.

In [ ]:
# Colab starts from a clean runtime; local and docs builds skip this setup.
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

if importlib.util.find_spec("google.colab") is not None:
    checkout = Path("/content/lczerolens")
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Xmaster6y/lczerolens.git", str(checkout)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(checkout), "pull", "--ff-only"], check=True)
    os.chdir(checkout)
    sys.path.insert(0, str(checkout))
    sys.path.insert(0, str(checkout / "src"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

In [ ]:
import chess
import torch
from torch import nn

from examples.decision_analysis_tutorial import load_fixture_evaluator
from lczerolens import EvaluationProvenance, LczeroEvaluator, LczeroModel
from lczerolens._codec import encode_move


class AlternativeFixture(nn.Module):
    def __init__(self):
        super().__init__()
        root = chess.Board()
        self.register_buffer("d4", torch.tensor(encode_move(root, chess.Move.from_uci("d2d4"))))
        self.register_buffer("e4", torch.tensor(encode_move(root, chess.Move.from_uci("e2e4"))))

    def forward(self, planes):
        policy = torch.zeros((planes.shape[0], 1858), device=planes.device)
        policy[:, self.d4] = 4.0
        policy[:, self.e4] = 1.0
        return policy, torch.full((planes.shape[0],), -0.1, device=planes.device)


model_a = load_fixture_evaluator().evaluator
model_b = LczeroEvaluator(
    LczeroModel(AlternativeFixture(), out_keys=["policy", "value"]),
    provenance=EvaluationProvenance(
        source="notebook-fixture", model_type="alternative-fixture-v1", network="fixture-b"
    ),
)

Both evaluators receive defensive copies of exactly the same boards. Keeping position identity and evaluator provenance in each frozen record makes later comparisons auditable.

In [ ]:
positions = []
for moves in ((), ("e2e4", "e7e5"), ("d2d4", "d7d5")):
    board = chess.Board()
    for move in moves:
        board.push_uci(move)
    positions.append(board)

evaluations_a = model_a.evaluate(positions)
evaluations_b = model_b.evaluate(positions)
rows = []
for index, (evaluation_a, evaluation_b) in enumerate(zip(evaluations_a, evaluations_b)):
    rows.append(
        {
            "position": index,
            "model_a_move": evaluation_a.policy.best_move.uci(),
            "model_b_move": evaluation_b.policy.best_move.uci(),
            "model_a_value": evaluation_a.value.value,
            "model_b_value": evaluation_b.value.value,
            "policy_changed": evaluation_a.policy.best_move != evaluation_b.policy.best_move,
        }
    )
rows

Persist the underlying observations rather than only the derived table. A downstream study can define its own aggregate metric without losing legal-policy probabilities, value origin, model identity, or position history.

In [ ]:
records_a = tuple(evaluation.record() for evaluation in evaluations_a)
records_b = tuple(evaluation.record() for evaluation in evaluations_b)
assert all(a.position == b.position for a, b in zip(records_a, records_b))
{
    "model_a": records_a[0].provenance.network,
    "model_b": records_b[0].provenance.network,
    "record_digests": [(a.digest()[:12], b.digest()[:12]) for a, b in zip(records_a, records_b)],
}